# Las compuertas cuánticas, paso a paso

Material de estudio de la sesión *Qiskit 101: tu primer circuito cuántico* (Qiskit Fall Fest 2026). Recorre, con la esfera de Bloch y experimentos, las compuertas que se ven en la presentación, hasta llegar al estado de Bell.

**Cómo usar este notebook:** ejecuta las celdas en orden con **Shift + Enter**. En cada compuerta:

1. Lee la **idea** y la analogía.
2. Mira su **matriz** y aprende a leerla.
3. Ejecuta y mira qué hace en la **esfera de Bloch**.
4. Haz el **experimento**: adivina antes de ejecutar.
5. Lee el recuadro verde **Idea clave**: lo esencial en pocas frases.

| Sección |
|---|
| 1. ¿Qué es un qubit? |
| 2. La esfera de Bloch: el mapa del qubit |
| 3. ¿Qué es una compuerta? |
| 4. X, Z e Y: los tres giros de 180° |
| 5. Hadamard: la puerta de entrada al mundo cuántico |
| 6. S y T: giros de fase más pequeños |
| 7. CNOT: la compuerta que entrelaza |
| 8. Tu primer circuito: H + CNOT |

In [ ]:
# Solo en Google Colab: quita el # de la línea siguiente y ejecuta esta celda primero
# %pip install -q "qiskit[visualization]"

In [ ]:
# Herramientas que usaremos en todo el notebook
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_bloch_multivector, plot_histogram

sampler = StatevectorSampler()


def ver_estado(qc):
    # Muestra el circuito, el estado resultante y su lugar en la esfera de Bloch
    estado = Statevector(qc)
    display(qc.draw("mpl"))
    display(estado.draw("latex"))
    return plot_bloch_multivector(estado)


def medir(qc, shots=1000):
    # Ejecuta el circuito muchas veces y cuenta los resultados
    qc_medido = qc.copy()
    qc_medido.measure_all()
    conteos = sampler.run([qc_medido], shots=shots).result()[0].data.meas.get_counts()
    print("Conteos:", conteos)
    return plot_histogram(conteos)


print("Listo")

---
## 1. ¿Qué es un qubit?

Un **bit** es un interruptor: está en `0` o en `1`.

Un **qubit** se describe con dos números, llamados **amplitudes**:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

- $|0\rangle$ y $|1\rangle$ son los dos estados "clásicos". Se escriben con esa forma, $|\ \rangle$, llamada **ket** (es la *notación de Dirac*).
- Al **medir**, sale `0` con probabilidad $|\alpha|^2$ y `1` con probabilidad $|\beta|^2$.
- Por eso siempre $|\alpha|^2 + |\beta|^2 = 1$: las probabilidades suman 100 %.

Como **vector**, el qubit es una columna de dos números:

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}, \qquad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix}, \qquad |\psi\rangle = \begin{pmatrix}\alpha\\ \beta\end{pmatrix}$$

In [ ]:
# Todo circuito empieza con sus qubits en |0⟩
qc = QuantumCircuit(1)
Statevector(qc).draw("latex")

<div class="alert alert-block alert-success">
<b>Idea clave · el qubit</b>

- *"Un bit es un interruptor: arriba o abajo. Un qubit es como una flecha que puede apuntar a cualquier lugar de una esfera."*
- *"Los números α y β no son probabilidades: son amplitudes. La probabilidad es su cuadrado."*
- Si te preguntan si un qubit está "en 0 y en 1 a la vez": es la forma popular de decirlo. Lo preciso es que está en una **combinación** con amplitudes, y **al medir se obtiene un solo resultado**.
</div>

---
## 2. La esfera de Bloch: el mapa del qubit

Cualquier estado de un qubit se puede dibujar como una **flecha** sobre una esfera:

| Lugar de la esfera | Estado | Al medir |
|---|---|---|
| Polo norte | $\lvert 0\rangle$ | siempre `0` |
| Polo sur | $\lvert 1\rangle$ | siempre `1` |
| Ecuador | superposiciones como $\lvert +\rangle$ y $\lvert -\rangle$ | 50 % / 50 % |

Dos ángulos ubican la flecha:
- **θ (theta):** qué tan "abajo" está la flecha, medido **desde el polo norte**. Decide las probabilidades.
- **ϕ (phi):** hacia dónde gira la flecha alrededor del eje vertical z. Es la **fase**: no cambia las probabilidades, pero sí importa (lo veremos con Z).

**La idea más importante de este notebook:**

> Toda compuerta de un qubit es un **giro** de la flecha en la esfera de Bloch.

In [ ]:
# El estado inicial |0⟩: la flecha apunta al polo norte
ver_estado(QuantumCircuit(1))

<div class="alert alert-block alert-success">
<b>Idea clave · la esfera de Bloch</b>

- *"La esfera es el mapa del qubit. Arriba es 0, abajo es 1, y en el ecuador está la moneda perfecta: 50/50."*
- *"Las compuertas mueven la flecha. Medir es preguntar '¿arriba o abajo?': cuanto más cerca del polo norte, más probable es el 0."*
</div>

---
## 3. ¿Qué es una compuerta?

Una compuerta cuántica es una **operación que transforma el estado** del qubit. Matemáticamente es una **matriz** que multiplica al vector:

$$|\psi'\rangle = U\,|\psi\rangle$$

### Cómo leer una matriz de compuerta (el truco más útil)

- La **primera columna** es lo que le pasa a $|0\rangle$.
- La **segunda columna** es lo que le pasa a $|1\rangle$.

### Dos reglas que cumplen todas las compuertas

1. **Son unitarias:** conservan la norma del vector, así que las probabilidades siempre siguen sumando 100 %.
2. **Son reversibles:** siempre se pueden deshacer. Esto las diferencia de las compuertas clásicas AND u OR: si AND da 0, no sabes si la entrada era 00, 01 o 10. La información se perdió.

Algunas compuertas, como X, Y, Z, H y CNOT, además son **hermitianas** (iguales a su conjugada transpuesta). Por eso, aplicarlas **dos veces** devuelve al inicio. S y T no lo son.

In [ ]:
# Verifiquemos las dos reglas con la compuerta X
qc = QuantumCircuit(1)
qc.x(0)
U = Operator(qc).data

print("Matriz de X:\n", U.real)
print("¿Unitaria?  U†·U = I ->", np.allclose(U.conj().T @ U, np.eye(2)))
print("¿Reversible? aplicar X dos veces = I ->", np.allclose(U @ U, np.eye(2)))

<div class="alert alert-block alert-success">
<b>Idea clave · qué es una compuerta</b>

- *"Una compuerta clásica cambia bits; una compuerta cuántica gira la flecha del qubit."*
- *"Para saber qué hace una compuerta, mira sus columnas: la primera dice qué le pasa al 0 y la segunda, qué le pasa al 1."*
- *"A diferencia de AND u OR, toda compuerta cuántica se puede deshacer. La información nunca se pierde... hasta que medimos."*
</div>

---
## 4. X, Z e Y: los tres giros de 180°

Las tres **compuertas de Pauli** son giros de media vuelta (180°), cada uno alrededor de un eje distinto.

### 4.1 Compuerta X: el NOT cuántico

$$X = \begin{pmatrix}0&1\\1&0\end{pmatrix}$$

Lectura por columnas: la primera columna es $(0, 1)$, o sea $|1\rangle$. Entonces $X|0\rangle = |1\rangle$ y $X|1\rangle = |0\rangle$.

**Cálculo a mano:**

$$X|0\rangle = \begin{pmatrix}0&1\\1&0\end{pmatrix}\begin{pmatrix}1\\0\end{pmatrix} = \begin{pmatrix}0\cdot1+1\cdot0\\1\cdot1+0\cdot0\end{pmatrix} = \begin{pmatrix}0\\1\end{pmatrix} = |1\rangle$$

En superposición, **intercambia las amplitudes**: $X(\alpha|0\rangle + \beta|1\rangle) = \beta|0\rangle + \alpha|1\rangle$.

En la esfera: **giro de 180° alrededor del eje X**. La flecha pasa del polo norte al polo sur.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
ver_estado(qc)

> **Antes de ejecutar, adivina:** ¿qué sale si aplicas X a $|+angle$ (un qubit que primero pasa por H)?

<details><summary><b>Respuesta</b> (ábrela después de ejecutar)</summary>

Queda **igual**: $X|+\rangle = |+\rangle$. El estado $|+\rangle$ está justo sobre el eje X, y girar 180° **alrededor de un eje** no mueve lo que está sobre el eje. Compruébalo en la celda siguiente.

</details>

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)   # prepara |+⟩
qc.x(0)   # aplica X
ver_estado(qc)

### 4.2 Compuerta Z: el cambio de fase

$$Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

- $Z|0\rangle = |0\rangle$: no lo toca.
- $Z|1\rangle = -|1\rangle$: le pone un **signo menos**.
- En superposición: $Z(\alpha|0\rangle + \beta|1\rangle) = \alpha|0\rangle - \beta|1\rangle$.

En la esfera: **giro de 180° alrededor del eje Z** (el vertical). Los polos no se mueven; lo que está en el ecuador da media vuelta: $|+\rangle \to |-\rangle$.

**El misterio de Z:** las probabilidades no cambian. $|\!-\!\beta|^2 = |\beta|^2$. Entonces, ¿para qué sirve?

In [ ]:
# Z sobre |+⟩: la flecha da media vuelta en el ecuador (|+⟩ → |−⟩)
qc = QuantumCircuit(1)
qc.h(0)
qc.z(0)
ver_estado(qc)

> **Antes de ejecutar, adivina:** si mides $|+angle$ y $Z|+angle = |-angle$, ¿salen histogramas distintos?

<details><summary><b>Respuesta</b> (ábrela después de ejecutar)</summary>

**No.** Los dos dan 50 % / 50 %. Medir solo pregunta "¿arriba o abajo?", y la fase es una dirección **horizontal** en la esfera. Por eso Z parece "invisible". Ejecuta la celda siguiente para comprobarlo.

</details>

In [ ]:
mas = QuantumCircuit(1); mas.h(0)
menos = QuantumCircuit(1); menos.h(0); menos.z(0)
display(medir(mas))
medir(menos)

**Cómo se vuelve visible la fase: interferencia.** Si después de Z aplicamos otra H, la fase se convierte en una diferencia que sí se mide:

$$H\,Z\,H = X$$

Una Z "escondida" entre dos H **se convierte en un NOT**.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.z(0)
qc.h(0)
medir(qc)   # siempre 1: la fase se volvió visible

### 4.3 Compuerta Y: X y Z al mismo tiempo

$$Y = \begin{pmatrix}0&-i\\i&0\end{pmatrix}$$

- $Y|0\rangle = i|1\rangle$
- $Y|1\rangle = -i|0\rangle$

Invierte el bit (como X) **y** cambia la fase (como Z). De hecho, $Y = iXZ$.

En la esfera: **giro de 180° alrededor del eje Y**. Al medir se comporta como X: $|0\rangle$ pasa a dar siempre `1`. La $i$ es una fase que solo se nota con interferencia.

In [ ]:
for inicial in ["0", "1"]:
    qc = QuantumCircuit(1)
    qc.y(0)
    print(f"Y|{inicial}⟩ =")
    display(Statevector.from_label(inicial).evolve(qc).draw("latex"))

<div class="alert alert-block alert-success">
<b>Idea clave · X, Y y Z</b>

- *"X, Y y Z son los tres giros de media vuelta: uno por cada eje de la esfera."*
- *"X es el NOT: cambia 0 por 1. Z no cambia las probabilidades, cambia la fase. Y hace las dos cosas."*
- *"La fase es como la dirección en que apunta una brújula en el ecuador: medir no la ve, pero la interferencia sí."*
- Pregunta típica: *"¿Para qué sirve Z si no cambia nada al medir?"*. Respuesta: *"Sirve para preparar la interferencia. Con dos H alrededor, Z se convierte en un NOT. Los algoritmos cuánticos juegan justamente con las fases."*
</div>

---
## 5. Hadamard: la puerta de entrada al mundo cuántico

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$$

Por columnas:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle, \qquad H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$$

- Convierte un estado definido (0 o 1) en una **superposición equilibrada**: 50 % / 50 %.
- $|+\rangle$ y $|-\rangle$ dan el mismo histograma; solo se diferencian en el **signo** (la fase).
- En la esfera: lleva los polos al **ecuador**. Es un giro de 180° alrededor del eje diagonal entre X y Z.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
display(ver_estado(qc))
medir(qc)

> **Antes de ejecutar, adivina:** si una H es «lanzar una moneda», ¿qué pasa con **dos H seguidas**? ¿50/50 otra vez?

<details><summary><b>Respuesta</b> (ábrela después de ejecutar)</summary>

**Siempre `0`.** $H\,H = I$: las amplitudes de $|1\rangle$ se **cancelan**.

$$|0\rangle \xrightarrow{H} \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle) \xrightarrow{H} \tfrac12(|0\rangle + |1\rangle) + \tfrac12(|0\rangle - |1\rangle) = |0\rangle$$

Una moneda real lanzada dos veces seguiría saliendo 50/50. Esta es la diferencia entre **superposición** y simple azar: la **interferencia**.

</details>

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)
medir(qc)

<div class="alert alert-block alert-success">
<b>Idea clave · Hadamard</b>

- *"La H lanza una moneda cuántica: deja al qubit en el ecuador de la esfera, 50/50."*
- *"Pero no es una moneda cualquiera: si la lanzas dos veces, vuelve a caer siempre igual. Las amplitudes se cancelan como ondas. Eso es la interferencia."*
- *"Casi todo algoritmo cuántico empieza con una capa de H: es la puerta de entrada a la superposición."*
</div>

---
## 6. S y T: giros de fase más pequeños

Son "primas" de Z: dejan $|0\rangle$ igual y le ponen una fase a $|1\rangle$, pero de menos de media vuelta.

| Compuerta | Matriz | $\lvert 1\rangle \to$ | Giro en el eje Z |
|---|---|---|---|
| Z | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | $-\lvert 1\rangle$ | 180° |
| S | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | $i\lvert 1\rangle$ | 90° |
| T | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | $e^{i\pi/4}\lvert 1\rangle$ | 45° |

- $T \cdot T = S$ y $S \cdot S = Z$.
- En el catálogo, T aparece como **"π/8"** por razones históricas (se puede escribir con ángulos de ±π/8 multiplicada por una fase global). Es la misma compuerta.
- **S y T no son hermitianas:** aplicarlas dos veces **no** devuelve al inicio.
- H, T y CNOT juntas forman un conjunto **universal**: combinándolas se puede construir cualquier operación cuántica.

In [ ]:
# Giros de fase en el ecuador: |+⟩ con T, con S y con Z
for nombre in ["t", "s", "z"]:
    qc = QuantumCircuit(1)
    qc.h(0)
    getattr(qc, nombre)(0)
    print(nombre.upper(), "sobre |+⟩:")
    display(plot_bloch_multivector(Statevector(qc)))

---
## 7. CNOT: la compuerta que entrelaza

La CNOT (*controlled-NOT*, NOT controlada) actúa sobre **dos qubits**:

- **Control (●):** no cambia nunca. "Decide" si se actúa.
- **Objetivo (⊕):** se invierte **solo si el control vale 1**.

$$\text{CNOT}|A, B\rangle = |A,\ A \oplus B\rangle$$

$\oplus$ es el **XOR** clásico (suma módulo 2): $0\oplus0=0,\ 0\oplus1=1,\ 1\oplus0=1,\ 1\oplus1=0$. Por eso el símbolo del objetivo es ⊕.

### El detalle de Qiskit: el orden de los qubits

Qiskit escribe los estados como $|q_1 q_0\rangle$: **el qubit 0 va a la derecha** (como las unidades en un número). Con `qc.cx(0, 1)`, el control es $q_0$ (derecha) y el objetivo es $q_1$ (izquierda):

| Entrada $\lvert q_1 q_0\rangle$ | ¿Control $q_0$ = 1? | Salida |
|---|---|---|
| $\lvert 00\rangle$ | no | $\lvert 00\rangle$ |
| $\lvert 01\rangle$ | **sí** | $\lvert 11\rangle$ |
| $\lvert 10\rangle$ | no | $\lvert 10\rangle$ |
| $\lvert 11\rangle$ | **sí** | $\lvert 01\rangle$ |

In [ ]:
# La tabla de la CNOT, comprobada con Qiskit
cnot = QuantumCircuit(2)
cnot.cx(0, 1)
display(cnot.draw("mpl"))

for entrada in ["00", "01", "10", "11"]:
    salida = Statevector.from_label(entrada).evolve(cnot)
    print(f"|{entrada}⟩  →  |{list(salida.to_dict())[0]}⟩")

> **Antes de ejecutar, adivina:** con cuatro posibles entradas clásicas, la CNOT solo copia o invierte bits. ¿Dónde está lo cuántico?

<details><summary><b>Respuesta</b> (ábrela después de ejecutar)</summary>

En lo que pasa cuando el **control está en superposición**: la CNOT no copia un 0 o un 1 ya decidido, sino la "moneda en el aire". El resultado es un estado **entrelazado**, que no se puede describir qubit por qubit. Es lo que construimos en la sección 8.

</details>

<div class="alert alert-block alert-success">
<b>Idea clave · la CNOT</b>

- *"La CNOT es un NOT con condición: el qubit de abajo se invierte solo si el de arriba vale 1."*
- *"Con entradas clásicas es un XOR que no pierde información. Su magia aparece cuando el control está en superposición: ahí crea entrelazamiento."*
- Si alguien viene de un libro de texto y ve la matriz CNOT "al revés": es por la convención de orden. Qiskit pone el qubit 0 a la derecha; muchos libros lo ponen a la izquierda. Ninguna de las dos está mal.
</div>

---
## 8. Tu primer circuito: H + CNOT = estado de Bell

Ahora juntamos todo. Un **circuito** es una secuencia de compuertas que se lee de **izquierda a derecha**.

Paso a paso (orden de Qiskit $|q_1 q_0\rangle$):

| Paso | Compuerta | Estado |
|---|---|---|
| 0 | — | $\lvert 00\rangle$ |
| 1 | H en $q_0$ | $\frac{1}{\sqrt2}(\lvert 00\rangle + \lvert 01\rangle)$: $q_0$ en superposición |
| 2 | CNOT ($q_0 \to q_1$) | $\frac{1}{\sqrt2}(\lvert 00\rangle + \lvert 11\rangle)$: **entrelazados** |

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
print("Después de H:")
display(Statevector(bell).draw("latex"))

bell.cx(0, 1)
print("Después de CNOT:")
display(Statevector(bell).draw("latex"))
bell.draw("mpl")

In [ ]:
medir(bell)

**Solo aparecen `00` y `11`.** Cada qubit, por separado, es una moneda perfecta (50/50). Pero los dos **siempre coinciden**.

**Entrelazado** significa que el estado no se puede escribir como "estado del qubit 1" × "estado del qubit 0". Prueba: si fuera $(a|0\rangle + b|1\rangle)(c|0\rangle + d|1\rangle)$, para que no aparezca $|01\rangle$ haría falta $a\cdot d = 0$, pero para que sí aparezcan $|00\rangle$ y $|11\rangle$ hacen falta $a\cdot c \neq 0$ y $b \cdot d \neq 0$. Es imposible.

> **Antes de ejecutar, adivina:** ¿qué cadenas aparecen si agregas una X en el qubit 1 **después** de la CNOT?

<details><summary><b>Respuesta</b> (ábrela después de ejecutar)</summary>

Solo `01` y `10`: ahora los qubits salen **siempre opuestos**. Es otro estado de Bell, $\frac{1}{\sqrt2}(|01\rangle + |10\rangle)$. Pruébalo modificando la celda de abajo.

</details>

In [ ]:
bell_opuesto = QuantumCircuit(2)
bell_opuesto.h(0)
bell_opuesto.cx(0, 1)
bell_opuesto.x(1)
medir(bell_opuesto)

<div class="alert alert-block alert-success">
<b>Idea clave · el estado de Bell</b>

- *"Con solo dos compuertas creamos entrelazamiento: la H lanza una moneda y la CNOT copia esa moneda, todavía en el aire, en el otro qubit."*
- *"Si miro un solo qubit, es una moneda perfecta. Si miro los dos, nunca se contradicen."*
- Lo que **no** significa: el entrelazamiento no permite enviar mensajes más rápido que la luz. Cada lado, por sí solo, ve resultados 100 % aleatorios; la correlación solo aparece al comparar los datos.
- *"Este es exactamente el circuito del tutorial Hello World de IBM, y el que vamos a ejecutar ahora en Jupyter."*
</div>

---
## Resumen: la tabla para tener a mano

| Compuerta | Qiskit | $\lvert 0\rangle \to$ | $\lvert 1\rangle \to$ | En la esfera | En una frase |
|---|---|---|---|---|---|
| X | `qc.x(q)` | $\lvert 1\rangle$ | $\lvert 0\rangle$ | 180° en el eje X | El NOT cuántico |
| Y | `qc.y(q)` | $i\lvert 1\rangle$ | $-i\lvert 0\rangle$ | 180° en el eje Y | NOT + cambio de fase |
| Z | `qc.z(q)` | $\lvert 0\rangle$ | $-\lvert 1\rangle$ | 180° en el eje Z | Cambia la fase |
| H | `qc.h(q)` | $\lvert +\rangle$ | $\lvert -\rangle$ | Polos al ecuador | Crea superposición |
| S | `qc.s(q)` | $\lvert 0\rangle$ | $i\lvert 1\rangle$ | 90° en el eje Z | Un cuarto de vuelta de fase |
| T | `qc.t(q)` | $\lvert 0\rangle$ | $e^{i\pi/4}\lvert 1\rangle$ | 45° en el eje Z | Un octavo de vuelta de fase |
| CNOT | `qc.cx(c, o)` | — | — | — | Invierte el objetivo si el control es 1; entrelaza |

## Autoevaluación

Responde sin mirar y luego abre la respuesta.

<details><summary>1. ¿Qué te dice la primera columna de la matriz de una compuerta?</summary>

En qué se convierte $|0\rangle$ al aplicar la compuerta.
</details>

<details><summary>2. ¿Por qué Z no cambia el histograma pero sí "hace algo"?</summary>

Cambia la fase relativa (el signo de $|1\rangle$). Medir no ve la fase, pero la interferencia sí: $HZH = X$.
</details>

<details><summary>3. ¿Qué estado deja H sobre |1⟩ y en qué se diferencia de H|0⟩?</summary>

$|-\rangle = \frac{1}{\sqrt2}(|0\rangle - |1\rangle)$. Da el mismo 50/50 que $|+\rangle$; solo cambia el signo.
</details>

<details><summary>4. ¿Por qué dos H seguidas dan siempre 0?</summary>

Porque $HH = I$: las amplitudes de $|1\rangle$ se cancelan (interferencia destructiva).
</details>

<details><summary>5. En Qiskit, con <code>cx(0, 1)</code>, ¿en qué se convierte |01⟩?</summary>

En $|11\rangle$. El control es $q_0$ (el bit de la derecha), que vale 1, así que se invierte $q_1$.
</details>

<details><summary>6. ¿Qué compuertas necesitas para entrelazar dos qubits?</summary>

Una que cree superposición (H) y una de dos qubits que los conecte (CNOT).
</details>

<details><summary>7. ¿Qué diferencia a las compuertas cuánticas de AND y OR?</summary>

Son reversibles (unitarias): nunca pierden información. AND y OR no se pueden deshacer.
</details>

<details><summary>8. ¿Por qué S y T no vuelven al inicio si las aplicas dos veces?</summary>

Porque no son hermitianas: $SS = Z$ y $TT = S$, no la identidad.
</details>

## Para seguir

- [`practica_en_vivo.ipynb`](../notebooks/practica_en_vivo.ipynb): el estado de Bell en Qiskit, en un simulador y en un computador cuántico real.
- [`hello-world_es.ipynb`](../notebooks/hello-world_es.ipynb): el circuito de Bell con Estimator y escalado a 100 qubits.
- [`get-started-with-qiskit_es.ipynb`](../notebooks/get-started-with-qiskit_es.ipynb): más compuertas (SWAP, Toffoli) y un semisumador cuántico.